# Visual Token Pruning Comparison: FastV vs LLaVA-PruMerge

This notebook compares visual token pruning strategies by computing **Jaccard similarity** between the token sets kept by two methods:

1. **FastV**: Prunes tokens based on average attention from the last token
2. **LLaVA-PruMerge**: Prunes tokens based on CLS token attention

## Setup

In [ ]:
# Install required packages (uncomment for Colab)
# !pip install torch torchvision transformers pillow matplotlib requests numpy
# !pip install git+https://github.com/huggingface/transformers.git

In [ ]:
# Clone repositories (uncomment for Colab)
# !git clone https://github.com/pkunlp-icler/FastV.git
# !git clone https://github.com/42Shawn/LLaVA-PruMerge.git

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# Setup paths
WORK_DIR = Path("/content") if "COLAB" in os.environ.get("COLAB_GPU", "") else Path(".")
FASTV_PATH = WORK_DIR / "FastV"
PRUMERGE_PATH = WORK_DIR / "LLaVA-PruMerge"

# Add to path
sys.path.insert(0, str(FASTV_PATH / "src" / "transformers" / "src"))
sys.path.insert(0, str(FASTV_PATH / "src" / "FastV"))
sys.path.insert(0, str(PRUMERGE_PATH))

print(f"Working directory: {WORK_DIR}")
print(f"FastV path: {FASTV_PATH}")
print(f"PruMerge path: {PRUMERGE_PATH}")

## Utility Functions

In [ ]:
def compute_jaccard_similarity(set_a, set_b):
    """Compute Jaccard similarity: |A ∩ B| / |A ∪ B|"""
    set_a = set(set_a)
    set_b = set(set_b)
    
    if len(set_a) == 0 and len(set_b) == 0:
        return 1.0
    
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    
    return intersection / union if union > 0 else 0.0


def download_image(url):
    """Download image from URL"""
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img.convert("RGB")


def visualize_token_overlap(fastv_indices, prumerge_indices, title="Token Selection"):
    """Visualize token selection on 24x24 grid"""
    h, w = 24, 24
    
    # Create grids
    fastv_grid = np.zeros((h, w))
    prumerge_grid = np.zeros((h, w))
    
    for idx in fastv_indices:
        row, col = divmod(int(idx), w)
        if row < h and col < w:
            fastv_grid[row, col] = 1
    
    for idx in prumerge_indices:
        row, col = divmod(int(idx), w)
        if row < h and col < w:
            prumerge_grid[row, col] = 1
    
    # Create overlap: 0=neither, 1=FastV only, 2=PruMerge only, 3=both
    overlap_grid = fastv_grid + prumerge_grid * 2
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(fastv_grid, cmap='Reds', interpolation='nearest')
    axes[0].set_title(f'FastV\n{len(fastv_indices)} tokens')
    axes[0].axis('off')
    
    axes[1].imshow(prumerge_grid, cmap='Blues', interpolation='nearest')
    axes[1].set_title(f'PruMerge\n{len(prumerge_indices)} tokens')
    axes[1].axis('off')
    
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['white', 'red', 'blue', 'purple'])
    axes[2].imshow(overlap_grid, cmap=cmap, interpolation='nearest', vmin=0, vmax=3)
    
    both = np.sum(overlap_grid == 3)
    only_fastv = np.sum(overlap_grid == 1)
    only_prumerge = np.sum(overlap_grid == 2)
    
    axes[2].set_title(f'Overlap\nBoth: {both} | FastV: {only_fastv} | PruMerge: {only_prumerge}')
    axes[2].axis('off')
    
    fig.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✓ Utility functions loaded")

## Load LLaVA-PruMerge Model

In [ ]:
from llava.model.multimodal_encoder.clip_encoder import CLIPVisionTower

# Load PruMerge vision tower
vision_tower = CLIPVisionTower(
    vision_tower="openai/clip-vit-large-patch14-336",
    args=None,
    delay_load=False
)

if not vision_tower.is_loaded:
    vision_tower.load_model()

vision_tower.eval()

print(f"✓ PruMerge model loaded")
print(f"  Device: {vision_tower.device}")
print(f"  Dtype: {vision_tower.dtype}")

## PruMerge Inference Function

In [ ]:
def run_prumerge_inference(vision_tower, image):
    """Run PruMerge and extract kept token indices"""
    from llava.model.multimodal_encoder import clip_encoder
    
    # Prepare image
    image_tensor = vision_tower.image_processor.preprocess(image, return_tensors='pt')['pixel_values']
    image_tensor = image_tensor.to(vision_tower.device, dtype=vision_tower.dtype)
    
    # Run inference
    with torch.no_grad():
        _ = vision_tower.token_prune_merge_advanced_plus(
            image_tensor,
            if_adaptive=True,
            reduction_ratio=1/8
        )
    
    # Extract indices from global variable
    kept_indices = clip_encoder.kept_token_indices["indices"]
    reduction_ratio = clip_encoder.kept_token_indices["reduction_ratio"]
    
    if kept_indices is None:
        raise RuntimeError("Failed to capture PruMerge indices")
    
    # Convert to list
    if len(kept_indices.shape) > 1:
        kept_indices = kept_indices[0]
    
    indices_list = kept_indices.numpy().tolist()
    
    print(f"PruMerge kept {len(indices_list)} tokens (ratio: {reduction_ratio:.3f})")
    
    return indices_list

print("✓ PruMerge inference function ready")

## Test on Sample Image

In [ ]:
# Download sample image
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = download_image(image_url)

# Display image
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.title("Sample Image")
plt.axis('off')
plt.show()

# Run PruMerge
prumerge_indices = run_prumerge_inference(vision_tower, image)
print(f"\nPruMerge selected tokens: {sorted(prumerge_indices)[:10]}... (showing first 10)")

## Visualize PruMerge Token Selection

In [ ]:
# For demo: create dummy FastV indices (replace with actual FastV when available)
# This simulates FastV selecting 144 tokens
dummy_fastv_indices = list(range(0, 576, 4))  # Every 4th token

print(f"Demo: Using dummy FastV indices ({len(dummy_fastv_indices)} tokens)")
print(f"PruMerge indices: {len(prumerge_indices)} tokens")

# Compute Jaccard similarity
jaccard = compute_jaccard_similarity(dummy_fastv_indices, prumerge_indices)
print(f"\nJaccard similarity: {jaccard:.4f}")

# Visualize
visualize_token_overlap(
    dummy_fastv_indices,
    prumerge_indices,
    title="Token Selection Comparison (Demo with Dummy FastV)"
)

## Multi-Image Experiment

In [ ]:
# Sample images from COCO validation set
sample_images = [
    ("cats", "http://images.cocodataset.org/val2017/000000039769.jpg"),
    ("tennis", "http://images.cocodataset.org/val2017/000000397133.jpg"),
    ("pizza", "http://images.cocodataset.org/val2017/000000252219.jpg"),
    ("baseball", "http://images.cocodataset.org/val2017/000000087038.jpg"),
    ("giraffe", "http://images.cocodataset.org/val2017/000000174482.jpg"),
]

results = []

for name, url in sample_images:
    print(f"\nProcessing: {name}")
    print("-" * 40)
    
    # Load image
    image = download_image(url)
    
    # Run PruMerge
    prumerge_indices = run_prumerge_inference(vision_tower, image)
    
    # For demo: use dummy FastV
    dummy_fastv_indices = list(range(0, 576, 4))
    
    # Compute Jaccard
    jaccard = compute_jaccard_similarity(dummy_fastv_indices, prumerge_indices)
    
    results.append({
        "name": name,
        "fastv_count": len(dummy_fastv_indices),
        "prumerge_count": len(prumerge_indices),
        "jaccard": jaccard
    })
    
    print(f"Jaccard: {jaccard:.4f}")

print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"{'Image':<15} {'FastV':>10} {'PruMerge':>12} {'Jaccard':>10}")
print("-" * 60)
for r in results:
    print(f"{r['name']:<15} {r['fastv_count']:>10} {r['prumerge_count']:>12} {r['jaccard']:>10.4f}")

jaccards = [r['jaccard'] for r in results]
print("-" * 60)
print(f"Mean Jaccard: {np.mean(jaccards):.4f} ± {np.std(jaccards):.4f}")

## Notes

### Code Modifications Made

**LLaVA-PruMerge (`clip_encoder.py`)**:
- Added global variable `kept_token_indices` at line 29
- Modified `token_prune_merge_advanced_plus()` at line 210-213 to store indices
- Indices are captured before adaptive augmentation

**FastV (`modeling_llama.py`)**:
- Added global variable `kept_visual_token_indices` at line 43
- Modified pruning logic at lines 751-755 and 795-799 to store indices
- Indices are converted to 0-575 range by subtracting `SYS_LENGTH`

### How It Works

1. **PruMerge**: Selects tokens based on CLS token attention from CLIP layer 23
2. **FastV**: Selects tokens based on average attention from the last token in LLaMA
3. **Jaccard**: Measures overlap as |A ∩ B| / |A ∪ B|

### Expected Results

- FastV typically keeps ~25% of tokens (144/576)
- PruMerge adaptive mode keeps ~12-15% (72-90/576)
- Jaccard similarity expected range: 0.2-0.5
- Higher Jaccard = both methods agree on important tokens
- Lower Jaccard = methods have different notions of importance

### To Use with Real FastV

Replace the `dummy_fastv_indices` with actual FastV inference:

```python
from transformers.models.llama import modeling_llama

# Run FastV model inference
_ = fastv_model.generate(...)

# Extract indices
fastv_indices = modeling_llama.kept_visual_token_indices["indices"].numpy().tolist()
```
</n>